# Service Hours dropdown chart

Is there a way to organize chart is to put all 3 day_types for 1 date in one chart
and let dropdown move from 1 month to another?

In [ ]:
import altair as alt
import gcsfs
import google.auth
import pandas as pd

import _new_operator_report_utils as utils
from update_vars import (
    DIGEST_DICT, PROCESSED_GCS, 
    abbrev_month, readable_dict, analysis_month
)

In [ ]:
analysis_name = "Alameda-Contra Costa Transit District"

In [ ]:
operator_hourly_summary_url = f"{PROCESSED_GCS}{DIGEST_DICT.hourly_day_type_summary}_{abbrev_month}.parquet"

operator_hourly_summary_df = pd.read_parquet(
    operator_hourly_summary_url,
    filesystem = gcsfs.GCSFileSystem(),
    filters=[[("Analysis Name", "==", analysis_name),
    ("Departure Hour", "<=", 24)]]
).reset_index(drop=True)

In [ ]:
test = operator_hourly_summary_df[operator_hourly_summary_df["Date"] == "07-2025"]

In [ ]:
date_dropdown = alt.binding_select(
    options=date_list,
    name="Dates: ",
)
xcol_param = alt.selection_point(
    fields=["Date"], value=date_list[0], bind=date_dropdown
)

In [ ]:
def create_bg_service_chart(df, y_col: str) -> alt.Chart:
    """
    Create a shaded background for the Service Hour Chart
    to differentiate between time periods.
    """
    specific_chart_dict = readable_dict.background_graph
    cutoff = pd.DataFrame(
        {
            "start": [0, 4, 7, 10, 15, 19],
            "stop": [3.99, 6.99, 9.99, 14.99, 18.99, 24],
            "Time Period": [
                "Owl:12-3:59AM",
                "Early AM:4-6:59AM",
                "AM Peak:7-9:59AM",
                "Midday:10AM-2:59PM",
                "PM Peak:3-7:59PM",
                "Evening:8-11:59PM",
            ],
        }
    )

    # Sort legend by time, 12am starting first.
    selection = alt.selection_point(fields=["Time Period"], bind="legend")

    chart = (
        alt.Chart(cutoff.reset_index())
        .mark_rect(opacity=0.15)
        .encode(
            x="start",
            x2="stop",
            #y=alt.value(0),
            #y2=alt.value(MAX_Y), # commenting this out means it layers better
            color=alt.Color(
                "Time Period:N",
                sort=(
                    [
                        "Owl:12-3:59AM",
                        "Early AM:4-6:59AM",
                        "AM Peak:7-9:59AM",
                        "Midday:10AM-2:59PM",
                        "PM Peak:3-7:59PM",
                        "Evening:8-11:59PM",
                    ]
                ),
                scale=alt.Scale(range=[*specific_chart_dict.colors]),
            ),
            opacity=alt.when(selection).then(alt.value(0.45)).otherwise(alt.value(0.15)),
            # when it's selected, gets darker
        ).add_params(selection)
    )

    return chart

In [ ]:
chart_dict = readable_dict.hourly_summary

In [ ]:
def make_basic_chart(df):
    chart = (
        alt.Chart(df)
        .mark_line(size=3)
        .encode(
            x=alt.X(
                "Departure Hour",
                title="Departure Hour",
                axis=alt.Axis(
                    labelAngle=-45,
                ),
            ),
            y=alt.Y(
                "N Trips",
                title="N Trips",
            ),
            color = alt.Color(
                "Day Type:N", 
                scale=alt.Scale(
                    domain=["Weekday", "Saturday", "Sunday"], 
                    range=[*chart_dict.colors]
                )
            )
        )
    )
    return chart

In [ ]:
#import _portfolio_charts
#bg = _portfolio_charts.create_bg_service_chart()
bg = create_bg_service_chart(test, "N Trips")

In [ ]:
selection = alt.selection_point(fields=["Day Type"], bind = "legend")
chart = (
    make_basic_chart(test)
    .encode(
        opacity=alt.when(selection).then(alt.value(1)).otherwise(alt.value(0.1)),
    ).add_params(selection)
)

#https://github.com/vega/altair/issues/772
(bg + chart).properties(   
    resolve=alt.Resolve(
        scale=alt.LegendResolveMap(color=alt.ResolveMode("independent")),
    )
)

In [ ]:
selection = alt.selection_point(fields=["Day Type"])
chart = (weekday_chart + sat_chart + sun_chart + bg)


In [ ]:



def create_hourly_summary(df: pd.DataFrame, day_type: str):

    chart_dict = readable_dict.hourly_summary
    df2 = df.loc[df["Day Type"] == "Saturday"]
    df2["Date"] = df["Date"].astype(str)

    date_list = list(df2["Date"].unique())

    date_dropdown = alt.binding_select(
        options=date_list,
        name="Dates: ",
    )
    xcol_param = alt.selection_point(
        fields=["Date"], value=date_list[0], bind=date_dropdown
    )

    chart = (
        (
            alt.Chart(df2)
            .mark_line(size=3)
            .encode(
                x=alt.X(
                    "Departure Hour",
                    title="Departure Hour",
                    axis=alt.Axis(
                        labelAngle=-45,
                    ),
                ),
                y=alt.Y(
                    "N Trips",
                    title="N Trips",
                ),
            )
        )
        .add_params(xcol_param)
        .transform_filter(xcol_param)
    )

    bg = _portfolio_charts.create_bg_service_chart()

    chart = (chart + bg).properties(
        resolve=alt.Resolve(
            scale=alt.LegendResolveMap(color=alt.ResolveMode("independent"))
        )
    )
    chart = _portfolio_charts.configure_chart(
        chart,
        width=400,
        height=250,
        title=f"{chart_dict.title} {day_type}",
        subtitle=chart_dict.subtitle,
    )

    return chart